# ViTreous — `train.ipynb`

Fine-tune ViT-S/16 (DeiT-S) on one dataset and export the weights.
**Swap datasets by changing a single string** (`DATASET`); everything
downstream derives from the dataset adapter (ARCHITECTURE.md §4).

Weights default to the **Kaggle output dir** (`/kaggle/working`); set
`PUSH_TO_HF = True` with an `HF_TOKEN` secret to also push to the Hub.

In [ ]:
# ─── Knobs — the only things you change to swap datasets/models ─────────
DATASET = "eurosat"          # or "oxford_pet" — proves the adapter layer
MODEL   = "vit_s16"          # registered timm ViT-S/16 (deit_small_patch16_224)
SEED    = 1234
EPOCHS  = 10
BATCH   = 64
LR      = 3e-4
DATA_ROOT = "/kaggle/input"  # Kaggle auto-attaches the dataset here
OUTPUT_DIR = "/kaggle/working"
PUSH_TO_HF = False           # set True + HF_TOKEN secret to push weights
HF_REPO = "<user>/vitreous-vit-s16-eurosat"

In [ ]:
# Install the vitreous core package (with the [ml] extra) from the repo branch.
# On Kaggle: enable Internet in the notebook settings. The package is the
# single code path shared with the live service — notebooks are thin shells.
import subprocess, sys
REPO = "https://github.com/<owner>/DragonHatchling.git"
BRANCH = "claude/explainable-vit-research-qadg2i"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"git+{REPO}@{BRANCH}#subdirectory=packages/core"], check=False)
# If the repo is attached as a Kaggle dataset/utility script instead, do:
#   %pip install -q -e /kaggle/input/vitreous/packages/core

In [ ]:
import torch, numpy as np
from vitreous.data import get_dataset, list_datasets
from vitreous.models import load_model

print("registered datasets:", list_datasets())
Adapter = get_dataset(DATASET)      # adapter class for the chosen dataset
adapter = Adapter()
spec = adapter.spec                 # DatasetSpec: classes, transforms, sources (§4)
print(f"{spec.display_name}: {spec.num_classes} classes")

In [ ]:
# Build the model with a fresh head sized to the dataset (§2).
device = 'cuda' if torch.cuda.is_available() else 'cpu'
lm = load_model(MODEL, spec, pretrained=True, num_classes=spec.num_classes)
model = lm.module.to(device)

In [ ]:
# Datasets + loaders from the adapter's own transforms/splits (§4).
from torch.utils.data import DataLoader, Dataset
from PIL import Image

class SampleDS(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.t = list(samples), transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        s = self.samples[i]
        img = Image.open(s.image).convert('RGB') if isinstance(s.image, str) else Image.fromarray(s.image)
        return self.t(img), s.label

train_s = adapter.load(DATA_ROOT, split='train')
val_s   = adapter.load(DATA_ROOT, split='val')
train_dl = DataLoader(SampleDS(train_s, adapter.augment()), batch_size=BATCH, shuffle=True, num_workers=2)
val_dl   = DataLoader(SampleDS(val_s, adapter.preprocess()), batch_size=BATCH, num_workers=2)

In [ ]:
# Standard fine-tuning loop.
torch.manual_seed(SEED)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
loss_fn = torch.nn.CrossEntropyLoss()
for epoch in range(EPOCHS):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); out = model(x); loss = loss_fn(out, y)
        loss.backward(); opt.step()
    # quick val accuracy
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(device)).argmax(1).cpu()
            correct += (p == y).sum().item(); total += len(y)
    print(f'epoch {epoch}: val_acc={correct/max(total,1):.4f}')

In [ ]:
# Export weights. Default: Kaggle output dir (always available).
import os, json
ckpt = os.path.join(OUTPUT_DIR, f'{MODEL}_{DATASET}.pt')
torch.save(model.state_dict(), ckpt)
metrics = {'val_acc': correct/max(total,1), 'epochs': EPOCHS, 'seed': SEED}
json.dump(metrics, open(os.path.join(OUTPUT_DIR, f'{MODEL}_{DATASET}.metrics.json'), 'w'))
print('saved', ckpt)

# Optional: push to the HF Hub (weights store, §2). Token via env/secret.
if PUSH_TO_HF:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_file(path_or_fileobj=ckpt, path_in_repo='pytorch_model.bin', repo_id=HF_REPO)
    print('pushed to', HF_REPO)